# Threshold Selection — F2 Optimisation

Load the trained XGBoost model, plot the precision-recall curve with F2 isocontours, and justify the chosen operating threshold.

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.metrics import precision_recall_curve, average_precision_score

import sys; sys.path.insert(0, '..')
from training.features.pandas_pipeline import engineer_features
from training.evaluate import f2, best_threshold_for_f2

MODEL_PATH = '../models/xgb.pkl'
DATA = '../data/raw/PS_20174392719_1491204439457_log.csv'

with open(MODEL_PATH, 'rb') as fh:
    artifact = pickle.load(fh)

pipeline = artifact['pipeline']
stored_threshold = artifact['threshold']
print(f'Stored threshold: {stored_threshold:.3f}  |  stored F2: {artifact["f2"]:.4f}')

In [ ]:
from sklearn.model_selection import train_test_split
from training.features.pandas_pipeline import FEATURE_COLS

df = pd.read_csv(DATA)
df = engineer_features(df)
X = df[FEATURE_COLS]; y = df['isFraud']
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
y_prob = pipeline.predict_proba(X_test)[:, 1]
print(f'Test set size: {len(y_test)}, fraud: {y_test.sum()}')

In [ ]:
prec, rec, thresholds = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)

# F2 isocontours
p_grid = np.linspace(0.01, 1, 200)
r_grid = np.linspace(0.01, 1, 200)
P, R = np.meshgrid(p_grid, r_grid)
F2_grid = (5 * P * R) / (4 * P + R)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PR curve with isocontours
ax = axes[0]
contours = ax.contourf(P, R, F2_grid, levels=np.arange(0.1, 1.0, 0.1), cmap='YlGn', alpha=0.35)
plt.colorbar(contours, ax=ax, label='F2')
ax.plot(prec, rec, 'b-', lw=2, label=f'XGBoost (AP={ap:.3f})')
ax.axvline(stored_threshold, color='red', ls='--', label=f'threshold={stored_threshold:.2f}')
ax.set_xlabel('Precision'); ax.set_ylabel('Recall')
ax.set_title('Precision-Recall Curve with F2 Isocontours')
ax.legend()

# Right: F2 vs threshold
ax2 = axes[1]
thresh_range = np.arange(0.01, 1.0, 0.01)
f2_scores = [f2(y_test, y_prob, t) for t in thresh_range]
ax2.plot(thresh_range, f2_scores, 'g-', lw=2)
best_idx = int(np.argmax(f2_scores))
ax2.axvline(thresh_range[best_idx], color='red', ls='--',
            label=f'best={thresh_range[best_idx]:.2f} F2={f2_scores[best_idx]:.3f}')
ax2.set_xlabel('Threshold'); ax2.set_ylabel('F2 Score')
ax2.set_title('F2 vs Decision Threshold')
ax2.legend()

plt.tight_layout()
plt.savefig('../docs/threshold_selection.png', dpi=120)
plt.show()
print(f'Optimal threshold: {thresh_range[best_idx]:.3f}  |  F2: {f2_scores[best_idx]:.4f}')